# Agent 365 - CSV Lander (manual upload only - fallback)

> **Fallback path — prefer [`Copilot_Agent365_Registry_Ingester.ipynb`](./Copilot_Agent365_Registry_Ingester.ipynb)** for
> production. The Ingester pulls Agent 365 live from Microsoft Graph (app-only, scheduled) and needs
> no CSV upload step. Use **this** Lander only when you can't grant the app-registration permissions
> the Ingester requires, or for one-off / evaluation runs from a static export.

Lands the **Agents 365** registry export into the Lakehouse Delta table `dbo.agents_365`, which the
Fabric dashboard reads via `FabricTable("agents_365")`. The two notebooks are **alternatives** — they
target the same Delta table, so running both in the same pipeline would just clobber each other.

Keeping Agent 365 in the Lakehouse (rather than a SharePoint URL in the report) keeps the Fabric
model **100% Lakehouse-sourced** — no gateway, no privacy-firewall, Direct Lake eligible.

**How to use:** drop the Agents 365 CSV at `Files/agent365/agents.csv` (or point `SOURCE_CSV` at a
OneLake shortcut), attach this notebook to the `<your-lakehouse>` Lakehouse, and Run all.
If the file is absent the dashboard's Agents 365 table simply loads empty (it is an optional source).

In [ ]:
# === CONFIG ===
SOURCE_CSV   = 'Files/agent365/agents.csv'   # drop the Agents 365 (MAC) export here, or a OneLake shortcut path
OUTPUT_TABLE = 'dbo.agents_365'              # Delta table read by the dashboard's Agents 365 table
WRITE_MODE   = 'overwrite'                   # 'overwrite' for full snapshots; 'append' for incremental

In [ ]:
# === LAND CSV -> Delta =========================================================
# Resilient to Agent 365 schema drift.
#
#   * every source column is kept, exactly as exported
#   * canonical names are added as COPIES, never renames, so no source column is
#     consumed (renaming 'Publisher' -> 'Agent creator' used to blank Publisher)
#   * header matching ignores case, spaces and punctuation, so 'Creator ID',
#     'Creator Id' and 'creator_id' all resolve
#   * a column that cannot be resolved is added as a typed null rather than
#     failing the run - the dashboard shows blank instead of breaking
#   * the match report below states what resolved, what fell back and what is
#     missing, so a rename is visible at ingestion instead of weeks later
import re
import notebookutils
from pyspark.sql import functions as F


def _exists(path):
    try:
        d = '/'.join(path.split('/')[:-1])
        return any(fi.name == path.split('/')[-1] for fi in notebookutils.fs.ls(d))
    except Exception:
        return False


if not _exists(SOURCE_CSV):
    print(f"Agent 365 export not found at {SOURCE_CSV} - nothing landed. "
          f"The dashboard's Agents 365 table will load empty (optional source).")
else:
    df = (spark.read
          .option('header', True)
          .option('multiLine', True)
          .option('escape', '"')
          .option('encoding', 'UTF-8')
          .csv(SOURCE_CSV))

    # Trim stray whitespace from headers before anything else looks at them.
    for c in df.columns:
        if c != c.strip():
            df = df.withColumnRenamed(c, c.strip())

    def _norm(s):
        """Fold a header to a comparable key: lowercase, alphanumerics only."""
        return re.sub(r'[^a-z0-9]', '', str(s).lower())

    lookup = {}
    for c in df.columns:
        lookup.setdefault(_norm(c), c)

    def _resolve(candidates):
        for cand in candidates:
            hit = lookup.get(_norm(cand))
            if hit is not None:
                return hit
        return None

    # Canonical target -> accepted source names, best first.
    # Add a new name to the front of a list when Microsoft renames a field; nothing
    # else needs to change. Matching is case/space/punctuation insensitive, so only
    # a genuinely new word needs adding.
    alias_plan = [
        ('Agent name',         ['Agent name', 'Name', 'Agent', 'Display name']),
        ('Supported in',       ['Supported in', 'Channel', 'Channels', 'Supported clients']),
        # 'Owner' is the tenant user who created the agent (blank for 1P/store agents).
        # 'Publisher' is the publishing company. Owner is preferred for 'Agent creator';
        # Publisher is a last resort and, because this is a copy, survives either way.
        ('Agent creator',      ['Agent creator', 'Owner', 'Created by', 'Developer Name', 'Publisher']),
        ('Agent type (A365)',  ['Agent type (A365)', 'Publisher Type', 'Type', 'Agent type']),
        ('Agent creator ID',   ['Agent creator ID', 'Creator Id', 'Owner Id', 'Created by id']),
        ('Agent description',  ['Agent description', 'Description']),
        ('Created in',         ['Created in', 'Platform', 'Source']),
        ('Last updated',       ['Last updated', 'Last Modified', 'Modified date']),
        ('Availability',       ['Availability', 'Status']),
        # 'Last Activity Date' was dropped from the export after July 2026 and
        # replaced by 'Last used'.
        ('Last Activity Date', ['Last Activity Date', 'Last used', 'Last activity', 'Last seen']),
        ('Active Users',       ['Active Users', 'Users', 'Monthly active users']),
        ('Total sessions',     ['Total sessions', 'Sessions']),
        ('Exception rate',     ['Exception rate', 'Error rate']),
    ]

    resolved, fallback, missing = [], [], []
    for target, candidates in alias_plan:
        src = _resolve([target] + candidates)
        if src is None:
            missing.append(target)
            continue
        if src != target:
            # COPY, never rename - the source column must survive for its own sake.
            df = df.withColumn(target, F.col(f'`{src}`'))
            fallback.append((target, src))
        else:
            resolved.append(target)

    # Columns the dashboard declares. Anything still absent is added as a typed null
    # so the Delta table always presents the same shape.
    canonical_cols = [
        'Agent name', 'Supported in', 'Date created', 'Agent creator', 'Publisher',
        'Agent type (A365)', 'Version', 'Availability', 'Agent creator ID',
        'Agent description', 'Created in', 'Last updated', 'Custom actions',
        'Title ID', 'Sensitivity',
        'Can read OneDrive and Sharepoint items', 'OneDrive and Sharepoint items',
        'Can read OneDrive files', 'OneDrive files', 'OneDrive sites',
        'Can read Sharepoint sites and files', 'Sharepoint files', 'Sharepoint sites',
        'Can extend to Graph connector', 'Graph connector details',
        'Can generate images using user prompt', 'Can use code interpreter',
        'Contains uploaded files', 'Uploaded files', 'Status',
        'Active Users', 'Total sessions', 'Exception rate', 'Last Activity Date',
        'Deployment', 'Run Time', 'Risks',
    ]

    added_null = []
    for col_name in canonical_cols:
        if col_name not in df.columns:
            df = df.withColumn(col_name, F.lit(None).cast('string'))
            added_null.append(col_name)

    # Canonical columns first, then every unrecognised source column - new fields
    # from a future export land automatically instead of being dropped.
    extras = [c for c in df.columns if c not in canonical_cols]
    df = df.select(*[F.col(f'`{c}`') for c in canonical_cols + extras])

    # Snapshot anchor from the export itself. 'Last updated' is guaranteed present
    # by the loop above, so this cannot fail; it is simply null if unparseable.
    as_of = df.agg(F.max(F.to_timestamp(F.col('`Last updated`')))).collect()[0][0]
    df = df.withColumn('Snapshot As Of', F.lit(as_of).cast('timestamp'))

    # ---- match report ---------------------------------------------------------
    print('Agent 365 schema match')
    print(f'  exact       : {len(resolved)}')
    if fallback:
        print(f'  via alias   : {len(fallback)}')
        for tgt, src in fallback:
            print(f'                {tgt!r} <- {src!r}')
    if missing:
        print(f'  NOT FOUND   : {len(missing)} (loaded as blank)')
        for tgt in missing:
            print(f'                {tgt}')
        print('                -> if a field above should exist, add its new export')
        print('                   name to alias_plan and re-run.')
    if extras:
        print(f'  extra cols  : {len(extras)} kept as-is')
    print(f'  snapshot as of: {as_of}')
    print(f'  rows: {df.count():,} | columns: {len(df.columns)}')

    (df.write.mode(WRITE_MODE)
       .option('overwriteSchema', 'true')
       .option('delta.columnMapping.mode', 'name')
       .format('delta').saveAsTable(OUTPUT_TABLE))
    print(f'wrote -> {OUTPUT_TABLE} ({WRITE_MODE})')
